In [0]:
from pyspark.sql.functions import explode, col, current_timestamp
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, ArrayType

# Ścieżki do kontenera Bronze w ADLS Gen2
RAW_HR_DIR = "abfss://bronze@adlsportfolioaw2026.dfs.core.windows.net/showcase_raw/hr_batch"
RAW_PAYROLL_DIR = "abfss://bronze@adlsportfolioaw2026.dfs.core.windows.net/showcase_raw/payroll_stream"
CHECKPOINT_PAYROLL = "abfss://bronze@adlsportfolioaw2026.dfs.core.windows.net/showcase_raw/checkpoints_payroll"

# ==========================================
# 0. PORZĄDEK W KATALOGU (Usuwamy starą kolizję tabeli)
# ==========================================
spark.sql("DROP TABLE IF EXISTS dbw_showcase.default.hr_bronze")
spark.sql("DROP TABLE IF EXISTS dbw_showcase.default.payroll_bronze")

# ==========================================
# 2. BATCH INGESTION (HR)
# ==========================================
print("⏳ Ładowanie danych HR (Batch) z ADLS Gen2...")

hr_schema = StructType([
    StructField("dept_id", IntegerType(), True),
    StructField("department_name", StringType(), True),
    StructField("employees", ArrayType(
        StructType([
            StructField("emp_id", IntegerType(), True),
            StructField("name", StringType(), True),
            StructField("city", StringType(), True),
            StructField("updated_at", StringType(), True)
        ])
    ), True)
])

hr_raw_df = spark.read.schema(hr_schema).json(RAW_HR_DIR)

hr_bronze_df = hr_raw_df.select(
    col("dept_id"),
    col("department_name"),
    explode(col("employees")).alias("employee") 
).select(
    col("dept_id"),
    col("department_name"),
    col("employee.emp_id").alias("emp_id"),
    col("employee.name").alias("emp_name"),
    col("employee.city").alias("city"),
    col("employee.updated_at").alias("updated_at"),
    current_timestamp().alias("ingested_at") 
)

# Zapis do tabeli Unity Catalog ze wskazaniem lokalizacji w naszym nowym kontenerze Bronze
hr_bronze_df.write.format("delta").mode("overwrite") \
    .option("path", "abfss://bronze@adlsportfolioaw2026.dfs.core.windows.net/tables/hr_bronze") \
    .saveAsTable("dbw_showcase.default.hr_bronze")

print("✅ Dane HR zapisane w warstwie Bronze jako tabela: dbw_showcase.default.hr_bronze")


# ==========================================
# 3. STREAMING INGESTION (Payroll) - Auto Loader
# ==========================================
print("⏳ Uruchamianie Auto Loadera dla logów Payroll z ADLS Gen2...")

payroll_stream_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", CHECKPOINT_PAYROLL + "_schema") 
    .load(RAW_PAYROLL_DIR)
    .withColumn("ingested_at", current_timestamp())
)

query = (payroll_stream_df.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_PAYROLL)
    .option("path", "abfss://bronze@adlsportfolioaw2026.dfs.core.windows.net/tables/payroll_bronze")
    .trigger(availableNow=True) 
    .toTable("dbw_showcase.default.payroll_bronze")
)

query.awaitTermination()
print("✅ Dane Payroll załadowane do tabeli: dbw_showcase.default.payroll_bronze")